# End-of-Day Mini Project 2 — Student Version (Google Colab)
## CIFAR-10 Image Classifier (End-to-End)

This notebook is a **student-learning version** of the attached mini project. It keeps **every topic and step** from the lab and adds short explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this project, you will build a complete image classification pipeline using a **Convolutional Neural Network (CNN)** on the **CIFAR-10** dataset. You will:
- load and augment data,
- define and train a CNN with a **StepLR learning-rate scheduler**,
- visualize training metrics,
- evaluate with classification metrics,
- plot a **confusion matrix**,
- and save the trained model using **`state_dict()`**.

### In this project, you will
- Load and augment the CIFAR-10 dataset
- Define a custom CNN using `nn.Module`
- Train the model with StepLR scheduler
- Visualize training loss and accuracy
- Evaluate the model using accuracy and F1-score (via classification report)
- Plot confusion matrix using seaborn
- Save the model using `torch.save(state_dict)`

### Estimated completion time
- **50 minutes**

### Runtime type (important)
- Set the Runtime type to: **T4 GPU**


# Task 0 (Colab setup): Imports + device check

### What you are learning
Before you train a CNN, you confirm your runtime and imports. This helps you avoid “silent” performance problems (e.g., running on CPU when you expected GPU) and prevents missing-library errors.


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# Task 1: Loading and augmenting CIFAR-10 dataset

### What you are learning
You are learning how to apply **data augmentation** to training images. Augmentation exposes the model to different “views” of the same class (flips/crops/color changes), which usually improves generalization on unseen test images. You will also visualize a batch to confirm augmentation is actually being applied.

### Steps (from the lab)
1. Import the necessary libraries.
2. Define class labels for CIFAR-10.
3. Define data augmentation for training set.
4. Define basic transform for test set (no augmentation).
5. Load CIFAR-10 training and test datasets with transforms.
6. Wrap datasets in DataLoaders.
7. Visualize one batch of augmented training images.
8. Review the output.


In [ ]:
# 1. Import the necessary libraries.
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch

# 2. Define class labels for CIFAR-10.
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# 3. Define data augmentation for training set.
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Random color adjustment
    transforms.ToTensor()
])

# 4. Define basic transform for test set (no augmentation).
test_transforms = transforms.Compose([
    transforms.ToTensor()
])

# 5. Load CIFAR-10 training and test datasets with transforms.
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=train_transforms)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=test_transforms)

# 6. Wrap datasets in DataLoaders.
trainloader = torch.utils.data.DataLoader(trainset, batch_size=8, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=8, shuffle=False)

print("Train samples:", len(trainset))
print("Test samples :", len(testset))


In [ ]:
# 7. Visualize one batch of augmented training images.
images, labels = next(iter(trainloader))  # Get one batch

plt.figure(figsize=(12, 5))
for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()  # Convert CHW → HWC
    plt.subplot(2, 4, i+1)
    plt.imshow(img)
    plt.title(f"Label: {classes[labels[i]]}")
    plt.axis('off')

plt.suptitle("Augmented CIFAR-10 Training Images", fontsize=16)
plt.tight_layout()
plt.show()


### Task 1 explanation (learning takeaways)
- Augmentation helps the model become **less sensitive** to small changes (position, lighting, orientation).
- The training set uses augmentation; the test set does **not**, so evaluation is fair and consistent.
- Visualizing a batch is a quick sanity check that your pipeline is working.


# Task 2: Designing a custom CNN using `nn.Module`

### What you are learning
You are learning how to define a more realistic CNN that includes:
- **Batch Normalization** (stabilizes training),
- **Max pooling** (reduces spatial size),
- **Dropout** (regularization),
- and **fully connected layers** that turn learned features into class scores.

You will also print the model, count parameters, and run a sample forward pass to verify output shape.

### Steps (from the lab)
1. Import the necessary libraries.
2. Define CNN architecture.
3. Instantiate model and print architecture.
4. Count total parameters.
5. Forward pass with one batch and print shapes.
6. Review the output.


In [ ]:
# 1. Import the necessary libraries.
import torch.nn as nn
import torch.nn.functional as F
import torch

# 2. Define CNN architecture.
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super(CIFAR10_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# 3. Instantiate model and print architecture.
model = CIFAR10_CNN()
print(" * Model Architecture:\n")
print(model)

# 4. Count total trainable parameters.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n * Total Trainable Parameters: {total_params}")

# 5. Forward pass with one batch.
sample_inputs, _ = next(iter(trainloader))
sample_outputs = model(sample_inputs)
print(f"\n * Sample Input Shape: {sample_inputs.shape}")
print(f" * Model Output Shape: {sample_outputs.shape}")


### Task 2 explanation (learning takeaways)
- Convolution layers learn feature detectors (edges → textures → shapes).
- BatchNorm can help gradients behave more consistently across layers.
- Dropout reduces overfitting by randomly zeroing some activations during training.
- The output shape should be `(batch_size, 10)` because CIFAR-10 has 10 classes.


# Task 3: Training the model with StepLR scheduler

### What you are learning
You are learning how to train a CNN using:
- `CrossEntropyLoss` for multi-class classification,
- `Adam` optimizer,
- and a **StepLR** scheduler that reduces the learning rate over time.

A scheduler can improve convergence by using a higher learning rate early (fast learning) and a lower learning rate later (fine-tuning).

### Steps (from the lab)
1. Import the necessary library.
2. Verify the device and move model to device.
3. Define loss function and optimizer.
4. Define StepLR scheduler (decay every 5 epochs by 0.5).
5. Train for 10 epochs while tracking loss and accuracy.
6. Print epoch stats including learning rate.
7. Review the output.


In [ ]:
# 1. Import the necessary library.
import torch.optim as optim

# 2. Verify the device and move model.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

# 3. Loss function and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Learning Rate Scheduler (decay every 5 epochs by 0.5).
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Initial LR:", scheduler.get_last_lr()[0])


In [ ]:
# 5. Training loop.
epochs = 10
train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 6. Stats (loss + accuracy)
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Step the scheduler once per epoch
    scheduler.step()

    avg_loss = running_loss / len(trainloader)
    accuracy = correct / total
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.4f} - LR: {scheduler.get_last_lr()[0]:.6f}")


### Task 3 explanation (learning takeaways)
- **Loss** measures how wrong predictions are; **accuracy** measures how often predictions are correct.
- StepLR gradually reduces learning rate, often improving later-stage training stability.
- Tracking metrics each epoch makes it easier to diagnose learning issues.


# Task 4: Visualizing training loss and accuracy

### What you are learning
Visual plots make learning behavior easier to interpret. You can quickly see whether:
- loss is decreasing,
- accuracy is increasing,
- training is stable,
- or if it looks stuck.

### Steps (from the lab)
1. Import the necessary library.
2. Plot training loss.
3. Plot training accuracy.
4. Add titles/labels/grids and show the figure.
5. Review the output.


In [ ]:
# 1. Import the necessary library.
import matplotlib.pyplot as plt

# 2–3. Plot training loss and accuracy.
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, marker='o')
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, color='green', marker='o')
plt.title("Training Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)

plt.suptitle(" Training Metrics")
plt.tight_layout()
plt.show()


### Task 4 explanation (learning takeaways)
- If loss decreases and accuracy increases, training is generally working.
- If loss is flat, you may need different hyperparameters (learning rate, model size, augmentation).
- Visuals help you spot instability (loss spikes) or slow progress.


# Task 5: Evaluating the model on test set

### What you are learning
You are learning how to evaluate on unseen data using:
- **accuracy**,
- and a **classification report** that includes precision, recall, and F1-score per class.

These metrics help you understand performance beyond “overall correctness,” especially if some classes are harder than others.

### Steps (from the lab)
1. Import necessary metrics.
2. Set model to evaluation mode and create storage lists.
3. Run inference on the test set (no gradients).
4. Calculate accuracy and print classification report.
5. Review the output.


In [ ]:
# 1. Import necessary metrics.
from sklearn.metrics import accuracy_score, classification_report

# 2. Set model to evaluation mode.
model.eval()
true_labels = []
predicted_labels = []

# 3. Run inference on the test set.
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        true_labels.extend(labels.numpy())
        predicted_labels.extend(preds.cpu().numpy())

# 4. Calculate accuracy and report.
test_accuracy = accuracy_score(true_labels, predicted_labels)
print(f" Test Accuracy: {test_accuracy:.4f}")

print("\n Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=classes))


In [ ]:
# (Optional but helpful) Quick summary F1 scores for the whole model.
# The classification report includes per-class F1 scores; this computes overall summary F1 values.
from sklearn.metrics import f1_score

macro_f1 = f1_score(true_labels, predicted_labels, average='macro')
weighted_f1 = f1_score(true_labels, predicted_labels, average='weighted')
print(f" Macro F1:    {macro_f1:.4f}")
print(f" Weighted F1: {weighted_f1:.4f}")


### Task 5 explanation (learning takeaways)
- `model.eval()` puts the model in inference mode (important for dropout/batchnorm behavior).
- `torch.no_grad()` saves memory and speeds up inference.
- The **classification report** gives precision/recall/F1 per class so you can see strengths and weaknesses class-by-class.


# Task 6: Plotting confusion matrix

### What you are learning
A confusion matrix shows how often each class is predicted as each other class. This is one of the best tools for spotting:
- which classes the model confuses most,
- and which classes it predicts confidently.

### Steps (from the lab)
1. Import necessary libraries.
2. Generate confusion matrix.
3. Plot using seaborn heatmap with class labels.
4. Review the output.


In [ ]:
# 1. Import necessary libraries.
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 2. Generate confusion matrix.
cm = confusion_matrix(true_labels, predicted_labels)

# 3. Plot using seaborn heatmap.
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, linewidths=0.5,
            linecolor='gray')

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title(" Confusion Matrix - CIFAR-10 Test Set", fontsize=14)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### Task 6 explanation (learning takeaways)
- The diagonal cells (top-left to bottom-right) represent correct predictions.
- Off-diagonal “hot spots” show common confusions (for example, cat vs dog).
- This helps you choose improvements (more data, different augmentation, deeper model).


# Task 7: Saving the best-performing model

### What you are learning
You are learning the standard PyTorch approach to saving models: save the **`state_dict()`**, which contains learned weights. This is portable and lets you reload the weights into the same architecture later.

> Note: The lab saves the trained model weights after training/evaluation. In a production workflow, you might track validation metrics each epoch and save only the best checkpoint.

### Steps (from the lab)
1. Import necessary library.
2. Create a directory for checkpoints if it doesn’t exist.
3. Define checkpoint path and save model state_dict.
4. Review the output.


In [ ]:
# 1. Import necessary library.
import os

# 2. Create a directory for checkpoints if it doesn’t exist.
os.makedirs("checkpoints", exist_ok=True)

# 3. Define path and save model.
checkpoint_path = "checkpoints/cifar10_cnn_best.pth"
torch.save(model.state_dict(), checkpoint_path)
print(f" Model state_dict saved to {checkpoint_path}")


# Lab review

1. What is the primary purpose of applying transformations like RandomHorizontalFlip, RandomCrop, and ColorJitter during training?  
A. To increase training speed  
B. To reduce model complexity  
C. To simulate different visual conditions and improve generalization  
D. To normalize image pixel values  

2. Saving only the model’s state_dict using torch.save() is more portable and flexible than saving the entire model object.  
A. True  
B. False  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

1. **C**  
2. **A (True)**

</details>
